In [21]:
from abc import ABC, abstractmethod
from time import perf_counter

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [22]:
MODELI = {
    "knn": KNeighborsClassifier,
    "stablo": DecisionTreeClassifier
}
def kreiraj_model(naziv):
    if naziv not in MODELI:
        raise ValueError("Nepoznat model")
    return MODELI[naziv]()
print(type(kreiraj_model("knn")))
print(type(kreiraj_model("stablo")))

<class 'sklearn.neighbors._classification.KNeighborsClassifier'>
<class 'sklearn.tree._classes.DecisionTreeClassifier'>


In [23]:
def mae(y, p):
    return sum(abs(a - b) for a, b in zip(y, p)) / len(y)
def mse(y, p):
    return sum((a - b) ** 2 for a, b in zip(y, p)) / len(y)
def accuracy(y, p):
    return sum(a == b for a, b in zip(y, p)) / len(y)
METRIKE = {
    "mae": mae,
    "mse": mse,
    "accuracy": accuracy
}
def izracunaj_metriku(y, p, metrika="accuracy"):
    if metrika not in METRIKE:
        raise ValueError("Nepoznata metrika")
    return METRIKE[metrika](y, p)
y = [3, 5]
p = [2, 7]
print(izracunaj_metriku(y, p, "mae"))
print(izracunaj_metriku(y, p, "mse"))
print(izracunaj_metriku([0, 1, 2], [0, 2, 2], "accuracy"))

1.5
2.5
0.6666666666666666


In [31]:
class Trener:
    def __init__(self):
        self.pretplatnici = []

    def pretplati(self, funkcija):
        self.pretplatnici.append(funkcija)

    def _objavi(self, dogadjaj, **podaci):
        for funkcija in self.pretplatnici:
            funkcija(dogadjaj, **podaci)

    def treniraj(self, model, X_train, y_train):
        self._objavi("pocetak_treninga")
        model.fit(X_train, y_train)
        self._objavi("kraj_treninga")
        return model


def log(dogadjaj, **podaci):
    print(dogadjaj, podaci)
trener = Trener()
trener.pretplati(log)
trener._objavi("proba")

proba {}


In [25]:
def mjeri_vrijeme(funkcija):
    def omotac(self):
        pocetak = perf_counter()
        rezultat = funkcija(self)
        print(f"Vrijeme: {perf_counter() - pocetak:.4f} s")
        return rezultat
    return omotac

In [26]:
class Eksperiment(ABC):
    @mjeri_vrijeme
    def pokreni(self):
        podaci = self.ucitaj()
        podaci = self.pripremi(podaci)
        model = self.treniraj(podaci)
        return self.evaluiraj(model, podaci)
    def pripremi(self, podaci):
        return podaci
    @abstractmethod
    def ucitaj(self):
        pass
    @abstractmethod
    def treniraj(self, podaci):
        pass
    @abstractmethod
    def evaluiraj(self, model, podaci):
        pass

In [27]:
class IrisEksperiment(Eksperiment):
    def __init__(self, naziv, metrika="accuracy"):
        self.naziv = naziv
        self.metrika = metrika
        self.trener = Trener()
    def ucitaj(self):
        return load_iris(return_X_y=True)
    def pripremi(self, podaci):
        X, y = podaci
        return train_test_split(X, y, test_size=0.2, random_state=15)
    def treniraj(self, podaci):
        X_train, X_test, y_train, y_test = podaci
        model = kreiraj_model(self.naziv)
        return self.trener.treniraj(model, X_train, y_train)
    def evaluiraj(self, model, podaci):
        X_train, X_test, y_train, y_test = podaci
        predikcije = model.predict(X_test)
        return izracunaj_metriku(y_test, predikcije, self.metrika)

In [28]:
for naziv in ["knn", "stablo"]:
    eksperiment = IrisEksperiment(naziv)
    eksperiment.trener.pretplati(log)
    tacnost = eksperiment.pokreni()
    print(f"{naziv}: {tacnost * 100:.2f}%")
    print()

pocetak_treninga {}
kraj_treninga {}
Vrijeme: 0.0087 s
knn: 100.00%

pocetak_treninga {}
kraj_treninga {}
Vrijeme: 0.0033 s
stablo: 93.33%

